In [1]:
# used the instructions of https://www.sparkcodehub.com/pyspark/use-cases/recommendation-systems
from pyspark.sql import SparkSession
from pyspark.ml.feature import StringIndexer
from pyspark.sql.functions import col
from pyspark.sql import functions as F
from pyspark.ml.evaluation import RegressionEvaluator

# SparkSession (like the wpo)
spark = SparkSession.builder \
    .master("local[*]") \
    .appName("AnimeRecommender") \
    .config("spark.ui.port", "4040") \
    .config("spark.hadoop.fs.defaultFS", "file:///") \
    .config("spark.driver.extraJavaOptions", "-Duser.name=admin") \
    .config("spark.driver.extraJavaOptions", "-Dsun.security.auth.login.config=C:/dev/null") \
    .getOrCreate()

In [2]:
#pre processed data
reviews_data = spark.read.csv("../data/preprocessed/reviews.csv", header=True, inferSchema=True)
anime_data = spark.read.csv("../data/preprocessed/animes.csv", header=True, inferSchema=True)

# The profiles need to be represented in numbers.
# Tried using cast like the intstructions but didn't work due to 
review_indexed = StringIndexer(inputCol="profile", outputCol="user_id")
reviews_data_indexed = review_indexed.fit(reviews_data).transform(reviews_data)

final_data = reviews_data_indexed.select(
    col("user_id").cast("integer"),
    col("anime_id").cast("integer"),
    col("score").cast("float")
)

final_data.show(7)

+-------+--------+-----+
|user_id|anime_id|score|
+-------+--------+-----+
|     32|   34096|  8.0|
|   1104|   34599| 10.0|
|   1825|   28891|  7.0|
|   3796|    2904|  9.0|
|   9589|    4181| 10.0|
|   9872|    2904| 10.0|
|    554|   16664|  6.0|
+-------+--------+-----+
only showing top 7 rows



In [3]:
###### make general functions (because it become a bit of a mess #######
from pyspark.ml.recommendation import ALS
from pyspark.ml.evaluation import RegressionEvaluator
from pyspark.ml.tuning import ParamGridBuilder, CrossValidator

# remove reviews with less than (number given)
def clean_sparse_data(input_data, min_reviews=3):
    print(f"Original Row Count: {input_data.count()}")

    # users
    valid_users = input_data.groupBy("user_id") \
        .agg(F.count("anime_id").alias("user_review_count")) \
        .filter(F.col("user_review_count") >= min_reviews)

    # anime
    valid_animes = input_data.groupBy("anime_id") \
        .agg(F.count("user_id").alias("anime_review_count")) \
        .filter(F.col("anime_review_count") >= min_reviews)

    cleaned_data = input_data.join(valid_users, on="user_id", how="inner") \
                         .join(valid_animes, on="anime_id", how="inner")

    cleaned_data  = cleaned_data .drop("user_review_count", "anime_review_count")
    print(f"Cleaned Row Count: {cleaned_data .count()}")
    
    return cleaned_data 


# Train with user bias, and check for different rank the best 
def train_and_evaluate_als(train_df, test_df):

    # Calculate the mean rating per user
    user_means = train_df.groupBy("user_id").agg(F.avg("score").alias("user_avg"))

    # Join this average back into the training and testing datasets
    train_with_means = train_df.join(user_means, on="user_id", how="left")
    test_with_means = test_df.join(user_means, on="user_id", how="left") 
    
    train_with_means = train_with_means.withColumn(
        "centered_score", 
        F.col("score") - F.col("user_avg")
    )
    train_with_means.select("user_id", "score", "user_avg", "centered_score").show(5)

    # ALS with the bias for the users ( im tired) #########################################
    als_bias = ALS(
        userCol="user_id",
        itemCol="anime_id",
        ratingCol="centered_score",   
        coldStartStrategy="drop"
    )

    param_grid = ParamGridBuilder() \
        .addGrid(als_bias.rank, [5, 10, 20]) \
        .addGrid(als_bias.regParam, [0.01, 0.1]) \
        .build()
    
    bias_evaluator = RegressionEvaluator(
        metricName="rmse",
        labelCol="centered_score",   
        predictionCol="prediction"
    )

    crossval = CrossValidator(
        estimator=als_bias,
        estimatorParamMaps=param_grid,
        evaluator=bias_evaluator, 
        numFolds=3
    )



    cv_model_biased = crossval.fit(train_with_means)
    best_biased_model = cv_model_biased.bestModel


    # Make predictions on the test data (which outputs a centered prediction) ###################
    predictions_with_bias = best_biased_model.transform(test_with_means)

    # Add the user_avg back to get the real 1-10 prediction
    final_predictions = predictions_with_bias.withColumn(
        "final_prediction", 
        F.col("prediction") + F.col("user_avg")
    )

    # Evaluate the FINAL RMSE on the original real 'score'
    final_evaluator = RegressionEvaluator(
        metricName="rmse",
        labelCol="score",               
        predictionCol="final_prediction" 
    )

    final_rmse = final_evaluator.evaluate(final_predictions)
    print(f"Final RMSE (With Mean Centering): {final_rmse}")

    return best_biased_model

In [4]:

cleaned_data = clean_sparse_data(final_data , min_reviews=3)


train_data, test_data = cleaned_data.randomSplit([0.8, 0.2], seed=42)


best_biased_model = train_and_evaluate_als(train_data, test_data)

Original Row Count: 130519
Cleaned Row Count: 82220
+-------+-----+-----------------+------------------+
|user_id|score|         user_avg|    centered_score|
+-------+-----+-----------------+------------------+
|      2| 10.0|5.846153846153846| 4.153846153846154|
|      4| 10.0|            6.925|             3.075|
|     14|  9.0|6.426229508196721|2.5737704918032787|
|     16|  9.0|5.764705882352941| 3.235294117647059|
|     20|  9.0|6.647619047619048| 2.352380952380952|
+-------+-----+-----------------+------------------+
only showing top 5 rows

Final RMSE (With Mean Centering): 2.1465675085203197


In [5]:
####### Bias #######

def get_global_mean(train_data):
    return train_data.agg(F.avg("score")).collect()[0][0]

def compute_user_biases(train_data, global_mean):
    return train_data.groupBy("user_id") \
        .agg(F.avg("score").alias("user_avg")) \
        .withColumn("user_bias", F.col("user_avg") - global_mean) \
        .select("user_id", "user_bias")

def compute_item_biases(train_data, global_mean):
    return train_data.groupBy("anime_id") \
        .agg(F.avg("score").alias("item_avg")) \
        .withColumn("item_bias", F.col("item_avg") - global_mean) \
        .select("anime_id", "item_bias")

def compute_all_biases(train_data):
    global_mean = get_global_mean(train_data)
    user_biases = compute_user_biases(train_data, global_mean)
    item_biases = compute_item_biases(train_data, global_mean)
    return global_mean, user_biases, item_biases

In [6]:
def train_and_evaluate_als(train_data, test_data):


    global_mean, user_biases, item_biases = compute_all_biases(train_data)

  
    train_with_biases = train_data.join(user_biases, on="user_id", how="left") \
                                .join(item_biases, on="anime_id", how="left")
    
    test_with_biases = test_data.join(user_biases, on="user_id", how="left") \
                              .join(item_biases, on="anime_id", how="left")
                              
    # Fill missing biases in test set with 0 (maybe change idk if correct and really nesecary)
    train_with_biases = train_with_biases.fillna(0, subset=["user_bias", "item_bias"])
    test_with_biases = test_with_biases.fillna(0, subset=["user_bias", "item_bias"])

    # Score - mu - bx - bi (score - gobal_bias - user_bias - item_bias
    train_with_biases = train_with_biases.withColumn(
        "centered_score", 
        F.col("score") - global_mean - F.col("user_bias") - F.col("item_bias")
    )

    # Train ALS 
    als_bias = ALS(
        userCol="user_id",
        itemCol="anime_id",
        ratingCol="centered_score",   
        coldStartStrategy="drop"
    )

    # check the best parameters for the model
    param_grid = ParamGridBuilder() \
        .addGrid(als_bias.rank, [5, 10, 15, 20, 35, 50, ]) \
        .addGrid(als_bias.regParam, [0.01, 0.1, 0.5]) \
        .build()
    
    bias_evaluator = RegressionEvaluator(
        metricName="rmse",
        labelCol="centered_score",   
        predictionCol="prediction"
    )

    crossval = CrossValidator(
        estimator=als_bias,
        estimatorParamMaps=param_grid,
        evaluator=bias_evaluator, 
        numFolds=3
    )


    cv_model_biased = crossval.fit(train_with_biases)
    best_biased_model = cv_model_biased.bestModel # best model found

    # Make predictions on the test data
    predictions_with_bias = best_biased_model.transform(test_with_biases)

    # Reconstruct the real prediction (Slide Math: Prediction + mu + bx + bi)
    final_predictions = predictions_with_bias.withColumn(
        "final_prediction", 
        F.col("prediction") + global_mean + F.col("user_bias") + F.col("item_bias")
    )

    # Evaluate the rmse in function of the score
    final_evaluator = RegressionEvaluator(
        metricName="rmse",
        labelCol="score",               
        predictionCol="final_prediction" 
    )

    final_rmse = final_evaluator.evaluate(final_predictions)
    print(f"RMSE (With all bias + hyper parameter): {final_rmse}")

    return best_biased_model

In [8]:
# data with profiles with 3 review or less removed
cleaned_data = clean_sparse_data(final_data, min_reviews=1)


train_data, test_data = cleaned_data.randomSplit([0.8, 0.2], seed=42)

# 3. Train the model
best_model = train_and_evaluate_als(train_data, test_data)
best_model.write().overwrite().save('/home/jovyan/work/models/user-item-better')

Original Row Count: 130519
Cleaned Row Count: 130519
RMSE (With all bias + hyper parameter): 2.1481888573215278
